# Calidad de los datos — Zona Metropolitana de Monterrey

**Este notebook no modifica nada.** Solo mide y reporta. Toda propuesta de
limpieza queda documentada al final, sin ejecutarse.

Fuente: `data/processed/atus_zmm.parquet` — los 18 municipios de la ZMM según el
Sistema Urbano Nacional, 2019-2024. Se genera con `uv run zona-atus` a partir del
consolidado nacional.

Esos 18 municipios son los únicos de Nuevo León que aparecen en ATUS: la
cobertura estatal de la encuesta coincide exactamente con la zona metropolitana.

**Ventaja sobre la base nacional: el panel está balanceado.** Los 18 municipios
están presentes los seis años, así que las series de tiempo sí son comparables
entre sí — a nivel nacional no lo son, porque la cobertura crece de 91 a 198
municipios.

La pregunta que organiza el reporte no es *"¿cuántos nulos hay?"* sino *"¿qué
información falta, y falta al azar?"*. Son preguntas muy distintas: casi no hay
`NaN`, y aun así solo una de cada cuatro filas está completa.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from geostats import rutas, zonas

d = pd.read_parquet(rutas.ATUS_ZMM)
N = len(d)
print(f"{N:,} registros x {len(d.columns)} columnas | {d.ANIO.min()}-{d.ANIO.max()} "
      f"| {d.CVE_MUN.nunique()} municipios")

## Estilo de las gráficas

Identidad GeoStats: Azul Prusia para los datos, Rojo profundo para el énfasis,
grafito para el texto y espacio negativo generoso.

Los colores de marca están pensados para impresión, no para marcas sobre fondo
claro, así que se validaron antes de usarlos: **Azul Prusia `#003153` no pasa**
(luminosidad 0.304 contra la banda 0.43–0.77, y croma 0.078 contra el piso 0.10;
a ese croma lee como gris). Se conserva su tono exacto —246°— y se sube la
luminosidad a 0.45. Los dos rojos pasan sin modificarse, pero entre ellos quedan
en ΔE 14.1 sobre un piso de 15, así que **nunca aparecen como series contiguas**.

In [ ]:
# --- Identidad GeoStats -------------------------------------------------------
# Azul Prusia es el color de datos de la marca, pero #003153 no sirve como marca
# sobre fondo claro: L=0.304 (la banda válida es 0.43-0.77) y croma 0.078 (piso
# 0.10), o sea que lee como gris. Se conserva su tono (246°) y se sube L a 0.45.
# Los dos rojos pasan sin tocarse, pero entre sí quedan en ΔE 14.1 (piso 15), así
# que nunca se usan como series contiguas.
AZUL     = "#005991"   # Azul Prusia ajustado -> datos
ROJO     = "#8B2C1A"   # Rojo profundo -> énfasis, títulos, cifras destacadas
OXIDO    = "#B15E2E"   # Rojo óxido -> detalle cálido
GRAFITO  = "#2C2C2C"   # texto y estructura
GRIS     = "#F2F2F2"   # fondos y rejilla
BLANCO   = "#FFFFFF"

# Rampa ordinal del mismo tono 246°, para dimensiones con orden. El croma se
# limita a 0.13 para no salirse del rango de la marca (0.078-0.133): la guía
# pide "sin saturación, sin neón". ΔL de 0.10 entre pasos, sobre el mínimo 0.06.
AZUL_RAMPA = ["#005991", "#1b77b8", "#4195d9"]

# La guía pide Montserrat (títulos), Cormorant Garamond (texto) y Roboto Mono
# (datos). Si no están instaladas, matplotlib baja al siguiente respaldo.
TITULAR = ["Montserrat", "Helvetica Neue", "Arial", "DejaVu Sans"]
TEXTO   = ["Cormorant Garamond", "EB Garamond", "Georgia", "DejaVu Serif"]
CIFRAS  = ["Roboto Mono", "Menlo", "DejaVu Sans Mono"]

plt.rcParams.update({
    "figure.dpi": 130,
    "figure.facecolor": BLANCO,
    "axes.facecolor": BLANCO,
    "savefig.facecolor": BLANCO,
    "font.family": "sans-serif",
    "font.sans-serif": TITULAR,
    "font.serif": TEXTO,
    "font.monospace": CIFRAS,
    "font.size": 9,
    "axes.edgecolor": GRIS,
    "axes.labelcolor": GRAFITO,
    "axes.labelsize": 9,
    "axes.titlecolor": ROJO,          # los títulos van en rojo profundo
    "axes.titlesize": 11.5,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.titlepad": 16,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": GRIS,
    "grid.linewidth": 1.0,
    "xtick.color": GRAFITO,
    "ytick.color": GRAFITO,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "legend.frameon": False,
    "legend.fontsize": 8.5,
    "lines.linewidth": 1.8,
})

def limpiar(ax, ejes=("top", "right", "left")):
    """Quita el marco y deja la rejilla como única referencia."""
    for lado in ejes:
        ax.spines[lado].set_visible(False)
    ax.tick_params(length=0)
    for etiqueta in ax.get_xticklabels() + ax.get_yticklabels():
        etiqueta.set_fontfamily("monospace")   # los datos, en monoespaciada
    return ax

def cifra(ax, x, y, texto, color=GRAFITO, **kw):
    """Etiqueta numérica directa, siempre en monoespaciada."""
    return ax.text(x, y, texto, family="monospace", fontsize=8.5,
                   color=color, **kw)

## 1. Los `NaN` mienten

Lo primero que uno hace es `df.isna().sum()`. En esta base, ese conteo es casi
vacío: solo tres columnas de texto tienen nulos.

In [ ]:
nulos = d.isna().sum()
nulos = nulos[nulos > 0].to_frame("nulos")
nulos["% del total"] = (nulos.nulos / N * 100).round(2)
nulos

Ese resultado sugiere una base casi perfecta. Es falso.

El INEGI **no usa `NaN`**: codifica la ausencia de información como un valor
numérico válido dentro del propio catálogo. Los códigos están documentados en
[`docs/diccionario_de_datos.md`](../docs/diccionario_de_datos.md).

In [ ]:
# Códigos centinela documentados en el diccionario del INEGI.
CENTINELAS = {
    "HORA":      {99: "no especificado"},
    "MINUTOS":   {99: "no especificado"},
    "DIA":       {32: "no especificado"},
    "DIASEMANA": {8: "no especificado"},
    "SEXO":      {1: "se fugó (sexo desconocido)"},
    "ALIENTO":   {6: "se ignora"},
    "CINTURON":  {9: "se ignora"},
    "EDAD":      {0: "se fugó", 99: "no especificado"},
}

filas = [
    {"campo": campo, "código": codigo, "significado": etiqueta,
     "registros": int((d[campo] == codigo).sum())}
    for campo, codigos in CENTINELAS.items()
    for codigo, etiqueta in codigos.items()
]
inventario = pd.DataFrame(filas)
inventario["% del total"] = (inventario.registros / N * 100).round(2)
inventario.sort_values("registros", ascending=False, ignore_index=True)

### La comparación

Cuánta información falta **según `isna()`** contra cuánta falta **de verdad**.

In [ ]:
def marca_centinela(campo):
    marca = pd.Series(False, index=d.index)
    for codigo in CENTINELAS.get(campo, {}):
        marca = marca | (d[campo] == codigo)
    return marca

def faltante_real(campo):
    return d[campo].isna() | marca_centinela(campo)

CAMPOS = ["CINTURON", "EDAD", "ALIENTO", "SEXO", "CALLE2"]
comparacion = pd.DataFrame({
    "código centinela": [marca_centinela(c).mean() * 100 for c in CAMPOS],
    "nulo explícito": [d[c].isna().mean() * 100 for c in CAMPOS],
}, index=CAMPOS)
comparacion["faltante real"] = comparacion.sum(axis=1)
comparacion = comparacion.sort_values("faltante real")
comparacion.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 3.6))
y = np.arange(len(comparacion))
hueco = 0.5  # el fondo separa los tramos; no se dibuja borde

ax.barh(y, comparacion["código centinela"], height=0.42,
        color=AZUL, label="Código centinela (isna no lo ve)")
ax.barh(y, comparacion["nulo explícito"], height=0.42,
        left=comparacion["código centinela"] + hueco,
        color=ROJO, label="Nulo explícito (lo ve isna)")

for i, v in enumerate(comparacion["faltante real"]):
    cifra(ax, v + 2, i, f"{v:.1f}%", va="center")

ax.set_yticks(y, comparacion.index)
ax.set_xlim(0, 100)
ax.set_xlabel("% de registros sin información", family="serif", fontsize=9.5)
ax.set_title("Casi todo el faltante está codificado, no nulo")
ax.xaxis.grid(True); ax.yaxis.grid(False)
leyenda = ax.legend(loc="lower right")
for t in leyenda.get_texts(): t.set_family("serif"); t.set_fontsize(9)
limpiar(ax)
plt.tight_layout()
plt.show()

Cada barra es el faltante total del campo, partido según cómo está expresado.
En `CINTURON`, `EDAD`, `ALIENTO` y `SEXO` el tramo rojo no existe: `isna()`
devuelve cero y aun así falta hasta el 72.3 % de la información. `CALLE2` es el
único caso donde el faltante sí es un nulo de verdad.

`HORA`, `MINUTOS`, `DIA` y `DIASEMANA` también traen centinelas, pero **en la ZMM
no aparecen en ningún registro**.

## 2. ¿Cuántos registros están realmente completos?

In [ ]:
CLAVE = ["CINTURON", "ALIENTO", "EDAD", "SEXO", "HORA", "DIA"]
matriz_falta = pd.DataFrame({c: faltante_real(c) for c in CLAVE})
cuantos = matriz_falta.sum(axis=1)

completas = int((cuantos == 0).sum())
print(f"Filas completas en los {len(CLAVE)} campos clave: {completas:,} de {N:,} ({completas/N:.1%})")

resumen = cuantos.value_counts().sort_index().to_frame("registros")
resumen.index.name = "campos faltantes"
resumen["% del total"] = (resumen.registros / N * 100).round(1)
resumen

**25.5 % de los registros tienen los seis campos clave** — apenas dos puntos
mejor que el 23.5 % nacional. Un análisis que use `dropna()` sobre estos campos
se quedaría con una cuarta parte de la base, y no con una cuarta parte
representativa: la sección 4 muestra por qué.

## 3. El faltante empeora con el tiempo

Aquí el panel balanceado importa: los 18 municipios están los seis años, así que
un cambio en la tasa es un cambio real de captura, no un efecto de cobertura.

In [ ]:
por_anio = pd.DataFrame(
    {c: faltante_real(c) for c in ["CINTURON", "EDAD", "ALIENTO", "SEXO"]}
).assign(ANIO=d.ANIO.values).groupby("ANIO").mean() * 100
por_anio.round(1)

In [ ]:
# Cuatro campos = cuatro paneles. Una sola serie por panel evita gastar cuatro
# tonos categóricos, que la paleta de marca no tiene, y deja leer cada curva
# contra el mismo eje.
campos = ["CINTURON", "EDAD", "ALIENTO", "SEXO"]
fig, ejes = plt.subplots(1, 4, figsize=(9.4, 2.9), sharey=True)

for ax, campo in zip(ejes, campos):
    serie = por_anio[campo]
    ax.plot(serie.index, serie.values, color=AZUL, marker="o", markersize=3.6,
            markerfacecolor=AZUL, markeredgecolor=BLANCO, markeredgewidth=1.2)
    cifra(ax, serie.index[-1], serie.iloc[-1], f" {serie.iloc[-1]:.0f}%",
          va="center", ha="left", color=ROJO)
    ax.set_title(campo, fontsize=10, pad=10, color=GRAFITO)
    ax.set_xticks([2019, 2024])
    ax.set_xlim(2018.7, 2025.3)
    ax.xaxis.grid(False)
    limpiar(ax)

ejes[0].set_ylim(0, 90)
ejes[0].set_ylabel("% sin información", family="serif", fontsize=9.5)
fig.suptitle("La captura se degrada año con año", x=0.005, ha="left",
             color=ROJO, fontsize=11.5, fontweight="bold")
plt.tight_layout(rect=(0, 0, 1, 0.90))
plt.show()

`EDAD` es el caso más claro: de 19.6 % a 35.9 % de faltante en seis años, casi el
doble. `ALIENTO` sube de 12.0 % a 18.5 %. `CINTURON` se mantiene arriba del 62 %
todo el periodo y `SEXO` es el único estable, cerca del 10 %.

Cualquier serie de tiempo sobre uso de cinturón, alcoholemia o edad del conductor
en la ZMM mide, en buena parte, **el deterioro del registro**.

## 4. El faltante no es aleatorio: depende de la gravedad

Si los datos faltaran al azar (MCAR), la tasa sería igual en accidentes fatales
y en los de solo daños.

In [ ]:
gravedad = d.CLASE.map({1: "Fatal", 2: "No fatal", 3: "Solo daños"})
por_gravedad = (
    pd.DataFrame({c: faltante_real(c) for c in ["ALIENTO", "EDAD", "SEXO", "CINTURON"]})
    .assign(gravedad=gravedad.values)
    .groupby("gravedad").mean().loc[["Fatal", "No fatal", "Solo daños"]] * 100
)
por_gravedad.round(1)

In [ ]:
# La gravedad tiene orden (fatal > no fatal > solo daños), así que se codifica
# con la rampa ordinal de un solo tono, no con tres colores categóricos.
campos = ["ALIENTO", "EDAD", "SEXO", "CINTURON"]
niveles = ["Fatal", "No fatal", "Solo daños"]

fig, ax = plt.subplots(figsize=(7.8, 3.9))
x = np.arange(len(campos))
ancho, hueco = 0.22, 0.02

for i, (nivel, color) in enumerate(zip(niveles, AZUL_RAMPA)):
    valores = [por_gravedad.loc[nivel, c] for c in campos]
    barras = ax.bar(x + (i - 1) * (ancho + hueco), valores, width=ancho,
                    color=color, label=nivel)
    for b, v in zip(barras, valores):
        cifra(ax, b.get_x() + b.get_width() / 2, v + 1.5, f"{v:.0f}%", ha="center")

ax.set_xticks(x, campos)
ax.set_ylim(0, 100)
ax.set_ylabel("% de registros sin información", family="serif", fontsize=9.5)
ax.set_title("Entre más grave el accidente, menos se sabe del conductor")
ax.xaxis.grid(False)
leyenda = ax.legend(ncols=3, loc="upper left", title="Gravedad del accidente")
leyenda.get_title().set_fontsize(8.5)
leyenda.get_title().set_color(GRAFITO)
for t in leyenda.get_texts(): t.set_family("serif"); t.set_fontsize(9)
limpiar(ax)
plt.tight_layout()
plt.show()

En accidentes **fatales** el aliento alcohólico se ignora en **35.0 %** de los
casos, contra 14.4 % en los de solo daños: **2.4 veces más seguido**. El
conductor se fugó en 24.3 % contra 9.5 %.

A diferencia de la base nacional, aquí **`CINTURON` también muestra el
gradiente** (86.0 % en fatales contra 71.8 % en solo daños), no es plano.

El mecanismo es **MNAR** (*missing not at random*): la probabilidad de que el
dato falte depende de aquello que se quiere estudiar. La explicación es
plausible — en un accidente fatal el responsable huye más seguido, y cuando huye
no hay a quién medirle el aliento ni preguntarle la edad.

**Consecuencia práctica:** eliminar las filas incompletas no es neutral. Borra
preferentemente los accidentes fatales con conductor fugado, que son justo el
subgrupo de mayor interés. Imputar tampoco es inocuo: los métodos estándar
(media, moda, MICE) suponen MAR, y aquí ese supuesto no se cumple.

> `EDAD` rompe la monotonía: «Solo daños» (27.9 %) queda por encima de «No fatal»
> (26.5 %). El patrón limpio es el de `ALIENTO` y `SEXO`. Si se cita el hallazgo,
> conviene citarlo sobre los extremos.

## 5. El problema mayor: no se captura igual en cada municipio

Esta es la diferencia más grande respecto al reporte nacional. Con un solo estado
y 18 municipios se puede ver de dónde viene realmente el faltante.

In [ ]:
d24 = d[d.ANIO == 2024]
tam = d24.groupby("NOM_MUN").size()
grandes = tam[tam > 500].index

por_municipio = (
    pd.DataFrame({c: faltante_real(c)[d.ANIO == 2024] for c in ["CINTURON", "ALIENTO"]})
    .assign(NOM_MUN=d24.NOM_MUN.values)
    .groupby("NOM_MUN").mean().loc[grandes] * 100
).join(tam.rename("accidentes_2024"))

por_municipio.sort_values("CINTURON", ascending=False).round(1)

In [ ]:
# Una sola magnitud -> un solo color. El rojo de la marca marca el caso que
# manda la historia; no es una segunda serie.
orden = por_municipio.sort_values("CINTURON")
colores = [ROJO if m == "Monterrey" else AZUL for m in orden.index]

fig, ax = plt.subplots(figsize=(7.4, 4.6))
y = np.arange(len(orden))
ax.barh(y, orden["CINTURON"], height=0.6, color=colores)

for i, (m, v) in enumerate(zip(orden.index, orden["CINTURON"])):
    cifra(ax, v + 1.5, i, f"{v:.0f}%", va="center",
          color=ROJO if m == "Monterrey" else GRAFITO)

ax.set_yticks(y, orden.index)
ax.set_xlim(0, 112)
ax.set_xticks([0, 25, 50, 75, 100])
ax.set_xlabel("% de accidentes con CINTURON = «se ignora», 2024",
              family="serif", fontsize=9.5)
ax.set_title("El uso de cinturón no se registra igual en cada municipio")
ax.xaxis.grid(True); ax.yaxis.grid(False)
limpiar(ax)
plt.tight_layout()
plt.show()

El rango va de **10.4 % en San Pedro Garza García a 100.0 % en Monterrey**. No es
una diferencia de conducta vial: es una diferencia de práctica administrativa.
San Pedro y San Nicolás capturan el dato en ~9 de cada 10 accidentes; en el otro
extremo hay 7 municipios por encima del 99 %.

**Monterrey está marcado en rojo no por ser el único en ese extremo, sino por su
peso: es el 45.4 % de los accidentes de la ZMM.** De sus 172,275 registros,
171,861 dicen «se ignora»: quedan **414 con dato real en seis años**. Los otros
6 municipios por encima del 99 % suman apenas el 5.9% de la base y no
mueven el agregado; Monterrey sí. Ese 72 % de faltante que vimos en la sección 1
es, en buena medida, un solo municipio.

In [ ]:
# ¿Es una política estable o cambia de un año a otro?
estabilidad = (
    d.assign(falta=faltante_real("CINTURON"))
     .pivot_table(index="NOM_MUN", columns="ANIO", values="falta", aggfunc="mean") * 100
)
estabilidad["rango"] = estabilidad.max(axis=1) - estabilidad.min(axis=1)
estabilidad.sort_values("rango", ascending=False).round(1)

La tabla muestra algo peor que un nivel alto: **inestabilidad**. Guadalupe pasa de
20.3 % en 2019 a 96.8 % en 2020. García va de 17.0 % a 99.0 % en tres años.
General Escobedo hace el camino inverso, de 99.9 % en 2022 a 42.3 % en 2024.

Son cambios de régimen administrativo, no de comportamiento. Comparar municipios
entre sí, o un municipio consigo mismo entre años, requiere controlar por la tasa
de captura.

## 6. Faltantes estructurales: no son errores

Algunos huecos son consecuencia lógica del diseño del formulario y no deben
tratarse como datos perdidos.

In [ ]:
print("SEXO=1 ('se fugó') y EDAD=0 ('se ignora porque se fugó') identifican")
print("exactamente los mismos registros:", bool(((d.SEXO == 1) == (d.EDAD == 0)).all()))
print(f"  registros afectados: {(d.SEXO == 1).sum():,}\n")

print("CALLE2 (vialidad secundaria) según el tipo de ubicación:")
etiqueta_urbana = d.URBANA.map({0: "0 suburbana", 1: "1 intersección", 2: "2 no intersección"})
tabla = pd.crosstab(etiqueta_urbana, d.CALLE2.isna(), normalize="index") * 100
tabla.columns = ["con CALLE2 (%)", "sin CALLE2 (%)"]
tabla.round(1)

`CALLE2` ausente en una fila con `URBANA = 0` (suburbana) o `URBANA = 2` (fuera
de intersección) es esperable: no hay segunda vialidad que registrar.

En cambio, **3,072 accidentes marcados como ocurridos en una intersección no
tienen segunda calle**. Ahí sí falta información, porque por definición una
intersección tiene dos vialidades. Ese es el subconjunto que vale la pena
revisar, no el 1.6 % agregado.

## 7. Texto

In [ ]:
texto = []
for c in ["CALLE1", "CALLE2", "CARRETERA"]:
    s = d[c]
    solo_espacios = s.notna() & (s.str.strip() == "")
    texto.append({
        "campo": c,
        "nulos": int(s.isna().sum()),
        "solo espacios": int(solo_espacios.sum()),
        "sin info (%)": round(float((s.isna() | solo_espacios).mean() * 100), 1),
        "valores únicos": int(s.nunique()),
        "con espacios sobrantes": int((s != s.str.strip()).sum()),
    })
pd.DataFrame(texto).set_index("campo")

La ZMM sale mejor librada que el resto del país en este punto: **no hay un solo
valor que sea solo espacios en blanco** (a nivel nacional hay 2,386 en
`CARRETERA`), y apenas 157 valores traen espacios sobrantes al inicio o al final.

`CARRETERA` está vacío en el 98.5 % de los registros porque solo aplica a
accidentes suburbanos, que aquí son el 1.7 % del total.

## 8. Coordenadas: válidas, pero muy concentradas

In [ ]:
en_zmm = d.LONGITUD.between(-101.0, -99.5) & d.LATITUD.between(25.0, 26.5)
print(f"Fuera del bounding box de la ZMM: {(~en_zmm).sum():,}")
print(f"Exactamente en (0, 0):            {((d.LONGITUD == 0) & (d.LATITUD == 0)).sum():,}")
print(f"bbox: lon {d.LONGITUD.min():.4f} a {d.LONGITUD.max():.4f}"
      f" | lat {d.LATITUD.min():.4f} a {d.LATITUD.max():.4f}")

decimales = d.LONGITUD.astype(str).str.split(".").str[1].str.len()
print("\nDecimales en LONGITUD (3 decimales ~ 110 m de error):")
print(decimales.value_counts().sort_index().to_string())

In [ ]:
puntos = d.groupby(["LONGITUD", "LATITUD"]).size().sort_values(ascending=False)
print(f"{len(puntos):,} coordenadas únicas para {N:,} accidentes\n")
for umbral in (1, 10, 100):
    cuantos = int((puntos > umbral).sum())
    peso = puntos[puntos > umbral].sum() / N
    print(f"  {cuantos:>6,} puntos con más de {umbral:>3} accidentes  ->  {peso:5.1%} de los registros")

print("\nPuntos más repetidos:")
for (lon, lat), k in puntos.head(3).items():
    sub = d[(d.LONGITUD == lon) & (d.LATITUD == lat)]
    calle = sub.CALLE1.value_counts().index[0]
    print(f"  ({lon}, {lat})  n={k:>4}  {sub.NOM_MUN.iloc[0]}  ·  {calle}")

Ninguna coordenada es inválida, pero la concentración es mucho mayor que a nivel
nacional: **72.3 % de los accidentes comparte coordenada exacta con otro**
(51.3 % nacional), y **365 puntos concentran el 17.9 % de la base** (contra 5.6 %
nacional). Solo hay 131,180 coordenadas distintas para 379,294 accidentes.

Tiene sentido: una traza urbana densa, geocodificada al nodo de cada
intersección. Los puntos más repetidos corresponden a corredores viales reales
—Eugenio Garza Sada, Manuel L. Barragán— no a un vertedero de centroides.

Aun así, para densidad de kernel o detección de *hotspots* esa duplicación exacta
infla los picos de forma artificial. Y como las AGEB urbanas del INEGI están
delimitadas por ejes de vialidad, estos puntos caen **sobre** las fronteras de
AGEB, no dentro de ellas.

Solo 866 registros tienen 3 decimales o menos (~110 m de error o peor).

## 9. Lo que sí está bien

Vale la pena separar los problemas reales de los imaginarios. La coherencia
interna de la base es impecable.

In [ ]:
muertos = ["CONDMUERTO", "PASAMUERTO", "PEATMUERTO", "CICLMUERTO", "OTROMUERTO"]
heridos = ["CONDHERIDO", "PASAHERIDO", "PEATHERIDO", "CICLHERIDO", "OTROHERIDO"]
vehiculos = ["AUTOMOVIL", "CAMPASAJ", "MICROBUS", "PASCAMION", "OMNIBUS", "TRANVIA",
             "CAMIONETA", "CAMION", "TRACTOR", "FERROCARRI", "MOTOCICLET",
             "BICICLETA", "OTROVEHIC"]
edad_valida = d.loc[~d.EDAD.isin([0, 99]), "EDAD"]

pruebas = {
    "TOTMUERTOS != suma de componentes": (d[muertos].sum(axis=1) != d.TOTMUERTOS).sum(),
    "TOTHERIDOS != suma de componentes": (d[heridos].sum(axis=1) != d.TOTHERIDOS).sum(),
    "CLASE=Fatal con cero muertos": ((d.CLASE == 1) & (d.TOTMUERTOS == 0)).sum(),
    "CLASE=Solo daños con víctimas": ((d.CLASE == 3) & (d.TOTMUERTOS + d.TOTHERIDOS > 0)).sum(),
    "URBANA y SUBURBANA marcadas a la vez": ((d.URBANA > 0) & (d.SUBURBANA > 0)).sum(),
    "Sin zona asignada": ((d.URBANA == 0) & (d.SUBURBANA == 0)).sum(),
    "Sin ningún vehículo involucrado": (d[vehiculos].sum(axis=1) == 0).sum(),
    "EDAD válida fuera del rango 12-98": (~edad_valida.between(12, 98)).sum(),
    "Llave (ANIO, EDO, MPIO, ID) duplicada": d.duplicated(["ANIO", "EDO", "MPIO", "ID"]).sum(),
    "Municipio fuera de la ZMM": (~d.CVE_MUN.isin(zonas.ZMM_MONTERREY)).sum(),
}
pd.Series({k: int(v) for k, v in pruebas.items()}, name="violaciones").to_frame()

Cero violaciones en las diez pruebas. Los totales cuadran con sus componentes, la
clasificación de gravedad es consistente con el conteo de víctimas, la llave no
tiene duplicados y ningún registro se coló de fuera de la zona metropolitana.

**El problema de esta base no es la integridad. Es quién llena el formato.**

## 10. Discrepancia con el diccionario

El diccionario del INEGI dice, sobre las coordenadas:

> `LONGITUD` — *Valor en formato decimal siempre positivo*
> `LATITUD` — *Valor en formato decimal siempre negativo*

Los datos son lo contrario, y los datos tienen razón: la ZMM está en longitud
negativa (oeste) y latitud positiva (norte). La observación del diccionario está
invertida. El inventario completo de discrepancias está en
[`docs/diccionario_de_datos.md`](../docs/diccionario_de_datos.md).

In [ ]:
d[["LONGITUD", "LATITUD"]].agg(["min", "max"]).round(4)

---

# Plan de limpieza propuesto

**Nada de esto está aplicado *en este notebook*, que solo mide.** Es el inventario
de decisiones a tomar, con la recomendación y su razón.

> **Ya implementado.** `geostats.limpieza` aplica estas decisiones como funciones
> explícitas y versionadas, y produce `atus_zmm_limpio.parquet`. Ver
> [`docs/limpieza.md`](../docs/limpieza.md). Este notebook se queda midiendo la
> base **sin limpiar**, que es lo que justifica la limpieza.

| # | Hallazgo | Acción propuesta | Por qué |
|---|---|---|---|
| 1 | Centinelas (`CINTURON=9`, `ALIENTO=6`, `EDAD∈{0,99}`, `SEXO=1`) | Convertir a `NA` en columnas paralelas (`CINTURON_NA`, …), **conservando el código original** | El código distingue *"se ignora"* de *"se fugó"*: son mecanismos distintos y colapsarlos a `NaN` pierde esa información |
| 2 | `SEXO=1` ⟺ `EDAD=0` (36,973 registros) | Derivar una bandera explícita `CONDUCTOR_FUGADO` | Es una variable con significado propio, y un predictor de gravedad, hoy escondida dentro de dos catálogos |
| 3 | MNAR por gravedad | **No usar `dropna()` ni imputación estándar** | El faltante depende de la gravedad; borrar sesga contra los accidentes fatales, e imputar bajo supuesto MAR introduce sesgo silencioso |
| 4 | Monterrey no registra `CINTURON` (414 datos reales en 172,275) | **Excluir `CINTURON` de cualquier análisis que incluya a Monterrey**, o restringirlo a los municipios que sí lo capturan | Con 45 % de la base sin el dato, un promedio de la ZMM describe a San Pedro y San Nicolás, no a la zona |
| 5 | Rango 10 %–100 % entre municipios, e inestable entre años | Reportar siempre la tasa de captura junto a cualquier comparación municipal o temporal | El rango mide práctica administrativa, no conducta vial |
| 6 | `EDAD` pasa de 19.6 % a 35.9 % de faltante (2019→2024) | Marcar la serie como no comparable entre años sin normalizar | El panel está balanceado, así que el deterioro es real y no un efecto de cobertura |
| 7 | 157 valores de texto con espacios sobrantes | `str.strip()` en una columna derivada | Fragmentan conteos (`"CENTRO"` y `"CENTRO "` cuentan como dos calles) |
| 8 | Nombres de vialidad sin normalizar | Normalizar mayúsculas y acentos **solo en una columna derivada** | Los acentos son parte del nombre real; destruirlos en la original es irreversible |
| 9 | 72.3 % de coordenadas duplicadas; 365 puntos = 17.9 % | Para KDE/*hotspots*, agregar por punto o aplicar *jitter* documentado | La duplicación exacta infla los picos de densidad |
| 10 | Puntos sobre ejes de vialidad | No asignar AGEB con un `within` simple | Las AGEB urbanas se delimitan por ejes de vialidad: los puntos caen en la frontera |
| 11 | ~~866 registros con ≤3 decimales~~ — **son 3** | Marcarlos y excluirlos de análisis a escala de intersección | **Corregido:** ese conteo mira solo `LONGITUD`. Con los dos ejes son 3, contra 2.3 esperados por azar — en la ZMM no hay coordenadas burdas |
| 12 | `CALLE2` ausente en 3,072 intersecciones | Marcar como faltante real, no estructural | Una intersección tiene dos vialidades por definición |

### Regla general

Los datos crudos viven en `data/raw/` y no se tocan. Cualquier limpieza produce
un artefacto nuevo en `data/processed/`, generado por código versionado y
reproducible — igual que hoy hacen `geostats.consolidar` y `geostats.zonas`.